# Exploratory Data Analysis — Ames House Prices

This notebook examines the deterministic cleaned dataset created in Phase 2. It is intentionally exploratory: reusable validation and cleaning remain in `src/`, and no modelling transformations are fitted here.

## Questions for this phase

- How is `SalePrice` distributed, and are large errors likely to matter?
- Which numerical and categorical attributes appear related to price?
- Which missing values are structural versus data-quality concerns?
- Which observations need investigation rather than automatic removal?

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

project_root = Path.cwd().resolve()
if not (project_root / 'data').exists():
    project_root = project_root.parent

sys.path.insert(0, str(project_root))
from src.data.preprocessing import load_processed_data

sns.set_theme(style='whitegrid', palette='deep')
pd.set_option('display.max_rows', 100)

data = load_processed_data(project_root / 'data/processed/train_clean.csv')
data.shape

## Dataset overview

We first confirm the feature mix. This determines whether the final preprocessing pipeline needs separate numerical and categorical branches.

In [ ]:
numeric_columns = data.select_dtypes(include='number').columns.tolist()
categorical_columns = data.select_dtypes(exclude='number').columns.tolist()

print(f'Rows: {len(data):,}')
print(f'Numerical columns: {len(numeric_columns)}')
print(f'Categorical columns: {len(categorical_columns)}')
data[['SalePrice', 'OverallQual', 'GrLivArea', 'Neighborhood']].head()

## Target distribution

**What we are analyzing:** the sale-price distribution and its tail.

**Why it matters:** a right-skewed target means a few expensive homes can dominate squared-error metrics. We will retain the price scale for our requested MAE/RMSE evaluation, then later assess whether a target transformation is justified by validation—not by this chart alone.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(data=data, x='SalePrice', kde=True, ax=axes[0])
axes[0].set_title('Sale price distribution')
sns.boxplot(data=data, x='SalePrice', ax=axes[1])
axes[1].set_title('Sale price spread and upper-tail observations')
plt.tight_layout()
print(data['SalePrice'].describe())
print(f"Skewness: {data['SalePrice'].skew():.2f}")

**Observation:** `SalePrice` is strongly right-skewed (about 1.88). The median is $163,000 while the upper tail reaches $755,000. This supports reporting both MAE and RMSE: RMSE will make costly misses visible.

## Numerical feature distributions

**What we are analyzing:** representative size, age, quality, and count fields.

**Why it matters:** very skewed, zero-heavy, or bounded variables should not be treated as interchangeable. Trees can tolerate non-linear scales, while linear regression may benefit from scaling within its pipeline.

In [ ]:
distribution_columns = ['GrLivArea', 'LotArea', 'TotalBsmtSF', 'YearBuilt', 'OverallQual', 'GarageCars']
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for column, axis in zip(distribution_columns, axes.ravel()):
    sns.histplot(data=data, x=column, kde=True, ax=axis)
    axis.set_title(column)
plt.tight_layout()

## Correlation with sale price

**What we are analyzing:** linear association between each numerical variable and the target.

**Why it matters:** correlation is not causation and does not reveal non-linear relationships, but it provides a transparent starting point for feature engineering and residual checks.

In [ ]:
target_correlations = (
    data.corr(numeric_only=True)['SalePrice']
    .drop('SalePrice')
    .sort_values(key=lambda values: values.abs(), ascending=False)
    .head(12)
)
plt.figure(figsize=(9, 6))
sns.barplot(x=target_correlations.values, y=target_correlations.index, hue=target_correlations.index, legend=False)
plt.title('Top numerical correlations with sale price')
plt.xlabel('Pearson correlation')
plt.ylabel('Feature')
plt.tight_layout()
target_correlations

**Observation:** `OverallQual` (~0.79) and `GrLivArea` (~0.71) have the strongest numerical association with price, followed by garage, basement, floor-area, and age measures. These will be central checks when we design logically meaningful aggregate features; we will not select features by correlation alone.

## Relationships and possible outliers

**What we are analyzing:** the relationship between above-ground living area, quality, and price.

**Why it matters:** a small number of unusually large, low-priced homes can disproportionately affect a linear model. They are investigated here, but retained unless there is a defensible data or business reason to exclude them.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.scatterplot(data=data, x='GrLivArea', y='SalePrice', hue='OverallQual', palette='viridis', ax=axes[0])
axes[0].set_title('Living area versus price')
sns.boxplot(data=data, x='OverallQual', y='SalePrice', ax=axes[1])
axes[1].set_title('Sale price by overall quality')
plt.tight_layout()

large_low_price = data.query('GrLivArea > 4000 and SalePrice < 300000')
print(f'Large, low-price homes requiring review: {len(large_low_price)}')
large_low_price[['GrLivArea', 'OverallQual', 'SalePrice', 'Neighborhood']]

**Observation:** living area and quality are positively related to price, but two very large homes sell below $300,000. They are valid recorded observations, so they remain in the data. Later model diagnostics will show whether they meaningfully harm generalization.

## Categorical patterns

**What we are analyzing:** the price distribution and sample size for each neighborhood.

**Why it matters:** location has a material relationship with price, and sparse categories can make estimates unstable. `Neighborhood` has only 25 levels, so one-hot encoding with unknown-category handling is appropriate.

In [ ]:
neighborhood_order = data.groupby('Neighborhood')['SalePrice'].median().sort_values().index
fig, axes = plt.subplots(1, 2, figsize=(18, 8), gridspec_kw={'width_ratios': [2, 1]})
sns.boxplot(data=data, y='Neighborhood', x='SalePrice', order=neighborhood_order, ax=axes[0])
axes[0].set_title('Sale price distribution by neighborhood')
sns.countplot(data=data, y='Neighborhood', order=neighborhood_order, ax=axes[1])
axes[1].set_title('Observations per neighborhood')
plt.tight_layout()
data.groupby('Neighborhood')['SalePrice'].agg(['median', 'count']).sort_values('median', ascending=False).head()

**Observation:** neighborhoods differ substantially in typical sale price (for example, `NridgHt` has a median near $315,000 while `MeadowV` is near $88,000). Some groups are small, so the pipeline must safely handle categories that are absent from a training fold or future request.

## Missing-value patterns

**What we are analyzing:** which fields are missing and how often.

**Why it matters:** high missingness in amenity-quality fields often represents the absence of that amenity. The model needs a consistent missing-value strategy, fitted only on training data.

In [ ]:
missing_percent = (data.isna().mean().mul(100).loc[lambda values: values > 0].sort_values(ascending=False))
plt.figure(figsize=(10, 7))
sns.barplot(x=missing_percent.values, y=missing_percent.index, hue=missing_percent.index, legend=False)
plt.title('Missing values by feature')
plt.xlabel('Missing values (%)')
plt.ylabel('Feature')
plt.tight_layout()
missing_percent.to_frame('missing_percent').round(1)

**Observation:** `PoolQC`, `MiscFeature`, `Alley`, and `Fence` are mostly missing, consistent with uncommon amenities. `LotFrontage` has moderate missingness (about 18%). In Phase 6, categorical values will receive a missing category and numerical values a training-set median imputation; we will not perform either operation in this notebook.

## Correlation between numerical features

**What we are analyzing:** relationships among the numerical features most associated with price.

**Why it matters:** strongly overlapping area and garage measurements can make linear-model coefficients less stable. A regular pipeline and model comparison will be more reliable than manually dropping fields from this chart.

In [ ]:
heatmap_columns = ['SalePrice', *target_correlations.head(8).index]
plt.figure(figsize=(10, 8))
sns.heatmap(data[heatmap_columns].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True)
plt.title('Correlation among target and leading numerical features')
plt.tight_layout()

# EDA Conclusions

1. `SalePrice` has a long right tail. We will use MAE, RMSE, and R² on the original dollar scale, and consider target transformation only through properly separated validation.
2. Overall quality, above-ground living area, basement area, garage capacity, bathrooms, and age are strong modelling candidates. Phase 4 will create only logical aggregates such as total square footage, total bathrooms, and house age.
3. Location matters substantially. `Neighborhood` has manageable cardinality and belongs in the categorical pipeline with `OneHotEncoder(handle_unknown='ignore')`.
4. Many missing values encode amenity absence; none will be filled in EDA. Train-fitted imputation belongs in Phase 6.
5. Two large, low-priced homes and several IQR flags deserve monitoring, but are not grounds for deletion. We will assess residuals after model fitting rather than silently removing valid cases.
6. Correlated size and garage fields argue for comparing models on the same pipeline instead of relying on a single coefficient-based story.